# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MuhammadAhmadIshtiaq/ml-internship-muhammadahmadishtiaq/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

**Signal check 1 — staleness (behind FlyRank's refresh flags).** Bucketed by `freshness_tier`, decline rate rises across the two buckets holding 98.8% of the data (0-30 days: 51.1%, n=20,480 → 91-180 days: 61.1%, n=9,171) — a real ~10pp gap on large samples. It reverses at 181+ days (47.1%), but that bucket is n=174 (0.6% of rows) — too small and noisy to trust over the main trend. **Verdict: CONFIRMED** (on the buckets that matter; the tiny reversal is flagged, not hidden).

**Signal check 2 — CTR-vs-position (behind the CTR-fix logic).** A page is `weak_ctr_for_position` if its CTR sits below the median CTR for its own `position_tier` (comparing against peers at the same rank, not overall). Decline rate: 50.3% (n=16,921) when CTR is normal-or-better for position, vs 59.3% (n=13,079) when CTR is weak for position — a clean 9pp gap on two large, balanced samples. **Verdict: CONFIRMED.**

**The rule, in plain words:** flag a page if it still gets real search visibility, is stale, AND its CTR is weak relative to its own position — both confirmed signals firing together, not just one.

**Reason code (one, fixed for every flagged row):** `stale_visible_weak_ctr`.
**Action label (one, fixed for every flagged row):** `flag_for_content_review`.

No forbidden columns touched: `trend_pct`, `trend_direction`, and the `_last_30d`/`_prev_30d` impression columns never appear in either signal check or the rule.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import numpy as np
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

# --- Signal 1: staleness (freshness_tier) vs decline rate ---
print("Signal 1 -- freshness_tier vs decline rate (n printed):")
signal1_table = df.groupby("freshness_tier")["is_declining_label"].agg(mean="mean", n="count")
print(signal1_table)
print("Verdict: CONFIRMED (on the two large buckets; 181+ reversal is noise, n=174)\n")

# --- Signal 2: CTR-vs-position vs decline rate ---
ctr_median_by_tier = df.groupby("position_tier")["ctr"].transform("median")
df["weak_ctr_for_position"] = (df["ctr"] < ctr_median_by_tier).astype(int)

print("Signal 2 -- weak_ctr_for_position vs decline rate (n printed):")
signal2_table = df.groupby("weak_ctr_for_position")["is_declining_label"].agg(mean="mean", n="count")
print(signal2_table)
print("Verdict: CONFIRMED\n")

# Confirm no forbidden columns were touched by either check
forbidden = {"trend_pct", "trend_direction", "impressions_last_30d", "impressions_prev_30d"}
used = {"freshness_tier", "ctr", "position_tier"}
print("Forbidden columns touched:", used & forbidden)


Signal 1 -- freshness_tier vs decline rate (n printed):
                    mean      n
freshness_tier                 
0-30            0.511377  20480
181+            0.471264    174
31-90           0.588571    175
91-180          0.611057   9171
Verdict: CONFIRMED (on the two large buckets; 181+ reversal is noise, n=174)

Signal 2 -- weak_ctr_for_position vs decline rate (n printed):
                           mean      n
weak_ctr_for_position                 
0                      0.502630  16921
1                      0.593088  13079
Verdict: CONFIRMED

Forbidden columns touched: set()


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

One rule, built only from the two confirmed signals: visible AND stale AND weak CTR for position. One numeric score for ranking, one fixed reason code, one fixed action label — no per-row branching.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

visible = (df["impressions_90d"] >= 100).astype(int)
stale = (df["freshness_tier"].isin(["91-180", "181+"])).astype(int)
weak_ctr = df["weak_ctr_for_position"]

rule_fires = (visible * stale * weak_ctr).astype(bool)

queue = pd.DataFrame({
    "content_id": df["content_id"],
    "client_id": df["client_id"],
    "is_declining_label": df["is_declining_label"],
})
queue["score"] = np.where(rule_fires, np.log1p(df["impressions_90d"]), 0.0)
queue["reason_code"] = np.where(rule_fires, "stale_visible_weak_ctr", "none")
queue["action"] = np.where(rule_fires, "flag_for_content_review", "no_action")

queue = queue.sort_values("score", ascending=False).reset_index(drop=True)
queue["rank"] = queue.index + 1

import os
os.makedirs("work/outputs", exist_ok=True)
queue.to_csv("work/outputs/baseline_action_score.csv", index=False)
print("Wrote work/outputs/baseline_action_score.csv --", len(queue), "rows,", int(rule_fires.sum()), "flagged")

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

base_rate = queue["is_declining_label"].mean()
print(f"\nBase rate: {base_rate:.1%}")
for k in [10, 20, 50]:
    p = precision_at_k(queue["score"], queue["is_declining_label"], k)
    print(f"precision@{k}: {p:.1%}  (vs {base_rate:.1%} base rate)")

# Save the baseline's metrics as receipts, per the card's note on what to commit
import json
metrics = {
    "base_rate": float(base_rate),
    "precision_at_10": float(precision_at_k(queue["score"], queue["is_declining_label"], 10)),
    "precision_at_20": float(precision_at_k(queue["score"], queue["is_declining_label"], 20)),
    "precision_at_50": float(precision_at_k(queue["score"], queue["is_declining_label"], 50)),
    "n_flagged": int(rule_fires.sum()),
}
with open("work/outputs/baseline_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)
print("\nWrote work/outputs/baseline_metrics.json")


Wrote work/outputs/baseline_action_score.csv -- 30000 rows, 2993 flagged

Base rate: 54.2%
precision@10: 70.0%  (vs 54.2% base rate)
precision@20: 65.0%  (vs 54.2% base rate)
precision@50: 52.0%  (vs 54.2% base rate)

Wrote work/outputs/baseline_metrics.json


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

Top-10 review: action, why it's there, and what would make it wrong — one line each.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

top10 = queue.head(10).copy()
top10["why_its_there"] = "Stale (91d+ since update), real search visibility, and CTR below the median for its ranking position."
top10["what_would_make_it_wrong"] = "If the low CTR is seasonal or the target keyword's intent doesn't match this page, not a real content problem."

top10[["rank", "content_id", "score", "reason_code", "action", "why_its_there", "what_would_make_it_wrong"]]


,rank,content_id,score,reason_code,action,why_its_there,what_would_make_it_wrong
0,1,content_5fe46e04994d,13.157182,stale_visible_weak_ctr,flag_for_content_review,"Stale (91d+ since update), real search visibil...",If the low CTR is seasonal or the target keywo...
1,2,content_36ff89c8214e,12.595063,stale_visible_weak_ctr,flag_for_content_review,"Stale (91d+ since update), real search visibil...",If the low CTR is seasonal or the target keywo...
2,3,content_c8e9d6ab9013,12.248552,stale_visible_weak_ctr,flag_for_content_review,"Stale (91d+ since update), real search visibil...",If the low CTR is seasonal or the target keywo...
3,4,content_a7427266c305,12.211617,stale_visible_weak_ctr,flag_for_content_review,"Stale (91d+ since update), real search visibil...",If the low CTR is seasonal or the target keywo...
4,5,content_91652435f57a,11.980370,stale_visible_weak_ctr,flag_for_content_review,"Stale (91d+ since update), real search visibil...",If the low CTR is seasonal or the target keywo...
5,6,content_f42eb861c6dd,11.934710,stale_visible_weak_ctr,flag_for_content_review,"Stale (91d+ since update), real search visibil...",If the low CTR is seasonal or the target keywo...
6,7,content_11fcfd65d94c,11.912265,stale_visible_weak_ctr,flag_for_content_review,"Stale (91d+ since update), real search visibil...",If the low CTR is seasonal or the target keywo...
7,8,content_97a86caf3a3d,11.902742,stale_visible_weak_ctr,flag_for_content_review,"Stale (91d+ since update), real search visibil...",If the low CTR is seasonal or the target keywo...
8,9,content_8b36799b7e44,11.859355,stale_visible_weak_ctr,flag_for_content_review,"Stale (91d+ since update), real search visibil...",If the low CTR is seasonal or the target keywo...
9,10,content_c1fe78bc4e37,11.806013,stale_visible_weak_ctr,flag_for_content_review,"Stale (91d+ since update), real search visibil...",If the low CTR is seasonal or the target keywo...


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Weak picks: rows in the top 10 flagged with high confidence but not labeled declining. Leakage check: re-confirm the rule and both signal checks never touched a forbidden column.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

weak_picks = top10[top10["is_declining_label"] == 0]
print(f"Weak picks in top 10: {len(weak_picks)} of 10")
if len(weak_picks) > 0:
    print(weak_picks[["rank", "content_id", "reason_code"]])

forbidden = {"trend_pct", "trend_direction", "impressions_last_30d", "impressions_prev_30d"}
used_anywhere = {"freshness_tier", "ctr", "position_tier", "impressions_90d"}
print("\nForbidden columns touched anywhere in signals or rule:", used_anywhere & forbidden, "(empty = clean)")

flag_like = [c for c in df.columns if "flag" in c.lower() or "score" in c.lower() or "decision" in c.lower()]
print("Existing product flags/scores in raw data:", flag_like, "(none exist to leak from)")


Weak picks in top 10: 3 of 10
   rank            content_id             reason_code
1     2  content_36ff89c8214e  stale_visible_weak_ctr
3     4  content_a7427266c305  stale_visible_weak_ctr
4     5  content_91652435f57a  stale_visible_weak_ctr

Forbidden columns touched anywhere in signals or rule: set() (empty = clean)
Existing product flags/scores in raw data: [] (none exist to leak from)


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.